# Part 4: Performance Arena

Chunking comparison, CAG simulation, CRAG impact and cost analysis.

In [ ]:
from prime_land.chat_service.cag_cache import CAGCache
from prime_land.chat_service.cag_service import CAGService
from prime_land.chat_service.crag_service import CRAGService
from prime_land.chat_service.rag_service import RAGService
from prime_land.config import CONFIG, get_path, load_faqs
from prime_land.evaluation.cag_eval import simulate_cag
from prime_land.evaluation.chunking_eval import evaluate_chunking
from prime_land.evaluation.cost_analysis import analyze_cost
from prime_land.evaluation.crag_eval import evaluate_crag
from prime_land.ingest_document.indexer import collection_name, get_client, open_store
from prime_land.providers import get_embeddings, get_llm

embeddings = get_embeddings()
llm = get_llm()
client = get_client()
stores = {strategy: open_store(client, embeddings, strategy) for strategy in CONFIG["chunking"]["strategies"]}

## Chunking strategy comparison

In [ ]:
chunking_table = evaluate_chunking(stores, embeddings, llm)
chunking_table

## CAG cache effectiveness

In [ ]:
strategy = CONFIG["retrieval"]["strategy"]
rag = RAGService(stores[strategy], embeddings, llm, strategy=strategy)
cache = CAGCache(embeddings, get_path("cache_dir"), CONFIG["cag"]["similarity_threshold"], CONFIG["cag"]["history_ttl_hours"])
cag = CAGService(rag, cache)
cag.warm_faqs(load_faqs())
cag_stats = simulate_cag(cag, load_faqs())
cag_stats

## CRAG correction impact

In [ ]:
crag = CRAGService(rag, llm)
crag_table, crag_summary = evaluate_crag(rag, crag, llm)
print(crag_summary)
crag_table

## Cost analysis (bonus)

In [ ]:
vector_count = client.count(collection_name(strategy)).count
analyze_cost(cag_stats, vector_count)

In [ ]:
client.close()